In [ ]:
# !pip install transformers torch

# SMS Spam Classification

Text-based spam detection enables platforms and service providers to automatically identify and filter unsolicited messages. By analyzing message content, organizations can reduce fraud, improve user experience, and maintain trust in communication services. Accurate classification also supports broader use cases such as threat detection and content moderation.

---

The dataset used in this laboratory is the **SMS Spam Collection v.1**, a public corpus of **5,574 English SMS messages** labeled as **ham (legitimate)** or **spam**. The dataset reflects a realistic class imbalance, with approximately **86.6% ham** and **13.4% spam**, and contains raw message text suitable for modern NLP workflows.

---

This laboratory focuses on **foundation-model-based text classification**, using a single dataset for consistency.

- **BERT (Bidirectional Encoder Representations from Transformers):**  
  A pretrained transformer model used as a foundation model to generate contextual text embeddings from raw SMS messages.

- **Perceptron (Linear Classifier):**  
  A single linear layer trained on top of BERT’s `[CLS]` representation to perform binary spam classification.

The model is trained using a stratified split and evaluated on a held-out validation set, highlighting the effectiveness of transfer learning with minimal task-specific modeling.


---
# Exploratory Data Analysis

Before training any models, we examine the dataset both statistically and qualitatively to establish a baseline understanding of the SMS messages. This step helps to:

- Understand the distribution and balance between **ham** and **spam** messages  
- Analyze message length, vocabulary usage, and basic linguistic patterns  
- Identify potential data quality issues such as empty messages or duplicates  
- Inform appropriate preprocessing decisions and modeling assumptions when using BERT  

In [ ]:
# Core data handling and utilities
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt

# Text modeling and deep learning
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


# Pretrained language models
from transformers import BertTokenizer, DistilBertModel

# Data splitting and evaluation
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [ ]:
# Load and prepare the SMS Spam dataset
# The dataset is expected to be located in the same directory as this notebook/script

df = pd.read_csv(
    "SMSSpamCollection.csv",
    sep="\t",
    header=None,
    names=["label", "text"]
)

# Normalize labels: ham → 0, spam → 1
df["label"] = df["label"].map({"ham": 0, "spam": 1})

# Inspect structure and sample data
df.info()
df.head()

In [ ]:
# Exploratory Data Analysis

# Class distribution
class_counts = df["label"].value_counts().sort_index()

print("Class distribution:")
print(class_counts)

# Plot spam vs ham distribution
plt.figure()
class_counts.plot(kind="bar")
plt.xticks([0, 1], ["Ham", "Spam"], rotation=0)
plt.xlabel("Message Type")
plt.ylabel("Number of Messages")
plt.title("SMS Spam vs Ham Distribution")
plt.show()

In [ ]:
# Train / Validation Split and BERT Tokenization

# Train / validation split (stratified)
X_train, X_val, y_train, y_val = train_test_split(
    df["text"].values,
    df["label"].values,
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

print(f"Train samples: {len(X_train)}")
print(f"Validation samples: {len(X_val)}")

# Initialize BERT tokenizer
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# PyTorch Dataset for SMS messages
class SpamDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            padding="max_length",
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "label": torch.tensor(self.labels[idx], dtype=torch.float)
        }

# Create datasets and dataloaders
train_dataset = SpamDataset(X_train, y_train, tokenizer)
val_dataset = SpamDataset(X_val, y_val, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

print("DataLoaders ready.")


In [ ]:
# DistilBERT + Perceptron Model 

# Automatic device selection
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

# DistilBERT + Perceptron model definition
class DistilBertPerceptron(nn.Module):
    def __init__(self, model_name="distilbert-base-uncased"):
        super().__init__()

        self.bert = DistilBertModel.from_pretrained(model_name)
        self.classifier = nn.Linear(self.bert.config.hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # DistilBERT uses the first token as sentence representation
        cls_embedding = outputs.last_hidden_state[:, 0, :]

        logits = self.classifier(cls_embedding)
        return logits.squeeze(-1)

# Model initialization
model = DistilBertPerceptron().to(device)

print("DistilBERT model initialized successfully.")

In [ ]:
# Training and Evaluation Loop (DistilBERT + Perceptron)

# Loss function and optimizer
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

# One training epoch
def train_epoch(model, dataloader):
    model.train()
    total_loss = 0

    for batch in dataloader:
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

# Evaluation
def evaluate(model, dataloader):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in dataloader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            logits = model(input_ids, attention_mask)
            preds = (torch.sigmoid(logits) > 0.5).long()

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return accuracy_score(all_labels, all_preds)

# Training loop
num_epochs = 3

for epoch in range(num_epochs):
    train_loss = train_epoch(model, train_loader)
    val_acc = evaluate(model, val_loader)

    print(
        f"Epoch {epoch + 1}/{num_epochs} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Validation Accuracy: {val_acc:.4f}"
    )


In [ ]:
# Confusion Matrix Evaluation

model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for batch in val_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        logits = model(input_ids, attention_mask)
        preds = (torch.sigmoid(logits) > 0.5).long()

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# Compute confusion matrix
cm = confusion_matrix(all_labels, all_preds)

# Plot confusion matrix
plt.figure()
plt.imshow(cm)
plt.title("Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.xticks([0, 1], ["Ham", "Spam"])
plt.yticks([0, 1], ["Ham", "Spam"])
plt.colorbar()

# Add values to the cells
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, cm[i, j], ha="center", va="center")

plt.show()


In [ ]:
# Multiple SMS inference example

sample_messages = [
    "Congratulations! You have won a free ticket. Call now!",
    "Hey, are we still meeting at 6 pm today?",
    "URGENT! Your account has been selected for a cash reward. Reply YES."
]

# Tokenize messages
encoding = tokenizer(
    sample_messages,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

input_ids = encoding["input_ids"].to(device)
attention_mask = encoding["attention_mask"].to(device)

# Run inference
model.eval()
with torch.no_grad():
    logits = model(input_ids, attention_mask)
    probabilities = torch.sigmoid(logits)

# Display results
for msg, prob in zip(sample_messages, probabilities):
    label = "Spam" if prob.item() > 0.5 else "Ham"
    print(f"Message: {msg}")
    print(f"Spam probability: {prob.item():.4f}")
    print(f"Prediction: {label}")
    print("-" * 60)


---
# Key Takeaways

The results demonstrate the effectiveness of a foundation-model-based approach when applied to a clean and well-defined spam classification task using the same dataset, split, and evaluation protocol.

- **DistilBERT + Perceptron** achieves very high performance with rapid convergence, requiring only a few training epochs to reach strong validation accuracy. This highlights the value of pretrained language models in capturing semantic patterns without manual feature engineering.

- The **Perceptron classifier** is sufficient for this task once contextual embeddings are provided by the foundation model, reinforcing the separation between representation learning and decision learning.

- The strong results are driven by **DistilBERT’s transformer-based architecture and pretraining on large-scale text corpora**, which enable robust semantic understanding and clear separation between spam and legitimate messages, even in the presence of class imbalance.


That´s all Folks !!!